# Assignment 4
Mohammad Rashiqul Alam  
malam25@illinois.edu  

# Part 1: Single-View Geometry

## Usage
This code snippet provides an overall code structure and some interactive plot interfaces for the *Single-View Geometry* section of Assignment 3. In [main function](#Main-function), we outline the required functionalities step by step. Some of the functions which involves interactive plots are already provided, but [the rest](#Your-implementation) are left for you to implement.

## Package installation
- In this code, we use `tkinter` package. Installation instruction can be found [here](https://anaconda.org/anaconda/tk).

# Common imports

In [12]:
%matplotlib tk
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from PIL import Image

# Provided functions

In [13]:
def get_input_lines(im, min_lines=3):
    """
    Allows user to input line segments; computes centers and directions.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        min_lines: minimum number of lines required
    Returns:
        n: number of lines from input
        lines: np.ndarray of shape (3, n)
            where each column denotes the parameters of the line equation
        centers: np.ndarray of shape (3, n)
            where each column denotes the homogeneous coordinates of the centers
    """
    n = 0
    lines = np.zeros((3, 0))
    centers = np.zeros((3, 0))

    plt.figure()
    plt.imshow(im)
    plt.show()
    print('Set at least %d lines to compute vanishing point' % min_lines)
    while True:
        print('Click the two endpoints, use the right key to undo, and use the middle key to stop input')
        clicked = plt.ginput(2, timeout=0, show_clicks=True)
        if not clicked or len(clicked) < 2:
            if n < min_lines:
                print('Need at least %d lines, you have %d now' % (min_lines, n))
                continue
            else:
                # Stop getting lines if number of lines is enough
                break

        # Unpack user inputs and save as homogeneous coordinates
        pt1 = np.array([clicked[0][0], clicked[0][1], 1])
        pt2 = np.array([clicked[1][0], clicked[1][1], 1])
        # Get line equation using cross product
        # Line equation: line[0] * x + line[1] * y + line[2] = 0
        line = np.cross(pt1, pt2)
        lines = np.append(lines, line.reshape((3, 1)), axis=1)
        # Get center coordinate of the line segment
        center = (pt1 + pt2) / 2
        centers = np.append(centers, center.reshape((3, 1)), axis=1)

        # Plot line segment
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], color='b')

        n += 1

    return n, lines, centers

In [14]:
def plot_lines_and_vp(im, lines, vp):
    """
    Plots user-input lines and the calculated vanishing point.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        lines: np.ndarray of shape (3, n)
            where each column denotes the parameters of the line equation
        vp: np.ndarray of shape (3, )
    """
    bx1 = min(1, vp[0] / vp[2]) - 10
    bx2 = max(im.shape[1], vp[0] / vp[2]) + 10
    by1 = min(1, vp[1] / vp[2]) - 10
    by2 = max(im.shape[0], vp[1] / vp[2]) + 10

    plt.figure()
    plt.imshow(im)
    for i in range(lines.shape[1]):
        if lines[0, i] < lines[1, i]:
            pt1 = np.cross(np.array([1, 0, -bx1]), lines[:, i])
            pt2 = np.cross(np.array([1, 0, -bx2]), lines[:, i])
        else:
            pt1 = np.cross(np.array([0, 1, -by1]), lines[:, i])
            pt2 = np.cross(np.array([0, 1, -by2]), lines[:, i])
        pt1 = pt1 / pt1[2]
        pt2 = pt2 / pt2[2]
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], 'g')

    plt.plot(vp[0] / vp[2], vp[1] / vp[2], 'ro')
    plt.show()

In [15]:
def get_top_and_bottom_coordinates(im, obj):
    """
    For a specific object, prompts user to record the top coordinate and the bottom coordinate in the image.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        obj: string, object name
    Returns:
        coord: np.ndarray of shape (3, 2)
            where coord[:, 0] is the homogeneous coordinate of the top of the object and coord[:, 1] is the homogeneous
            coordinate of the bottom
    """
    plt.figure()
    plt.imshow(im)

    print('Click on the top coordinate of %s' % obj)
    clicked = plt.ginput(1, timeout=0, show_clicks=True)
    x1, y1 = clicked[0]
    # Uncomment this line to enable a vertical line to help align the two coordinates
    # plt.plot([x1, x1], [0, im.shape[0]], 'b')
    print('Click on the bottom coordinate of %s' % obj)
    clicked = plt.ginput(1, timeout=0, show_clicks=True)
    x2, y2 = clicked[0]

    plt.plot([x1, x2], [y1, y2], 'b')

    return np.array([[x1, x2], [y1, y2], [1, 1]])

# Your implementation

In [16]:
def get_vanishing_point(lines):
    """
    Solves for the vanishing point from a set of user-input lines.
    Each column of `lines` is a homogeneous line [a,b,c]^T so that a*x + b*y + c = 0.
    We solve L^T v = 0 in the least-squares sense and return a normalized homogeneous v.
    Inputs:
        lines: (3, n) ndarray
    Returns:
        vp: (3,) ndarray  (homogeneous; vp[2] != 0)
    """
    # SVD on L^T to find the 1D nullspace
    # L^T v = 0  -> v is right-singular vector associated with smallest singular value
    U, S, Vt = np.linalg.svd(lines.T)
    vp = Vt[-1, :]
    # normalize so that vp[2] = 1 when possible
    if abs(vp[2]) > 1e-12:
        vp = vp / vp[2]
    return vp

In [17]:
def get_horizon_line(vp1, vp2):
    """
    Ground horizon line through the two horizontal vanishing points.
    Inputs:
        vp1, vp2: each (3,) homogeneous vanishing points with vp[2] possibly == 1
    Returns:
        h: (3,) homogeneous line parameters (a,b,c) with ||(a,b)|| = 1
    """
    # Ensure homogeneous scaling by making vp[2] = 1 when possible
    if abs(vp1[2]) > 1e-12: 
        vp1 = vp1 / vp1[2]
    if abs(vp2[2]) > 1e-12: 
        vp2 = vp2 / vp2[2]
    # line through the two VPs
    h = np.cross(vp1, vp2)       
    # Normalize so that a^2 + b^2 = 1
    ab_norm = np.linalg.norm(h[:2])
    # Avoid division by zero
    if ab_norm > 0:
        h = h / ab_norm
    return h

In [18]:
def plot_horizon_line(im, horizon_line):
    """
    Plots the horizon line over the image.
    Inputs:
        im: (H,W,3) image
        horizon_line: (3,) line parameters (a,b,c) with a^2 + b^2 = 1
    """
    a, b, c = horizon_line
    H, W = im.shape[0], im.shape[1]

    # Compute intersections with left/right borders (x=0 and x=W-1)
    # For b != 0, y = -(a*x + c)/b; handle near-vertical horizon robustly
    xs = np.array([0, W-1], dtype=float)
    ys = np.full_like(xs, np.nan, dtype=float)
    # 1e-12 is arbitrary small threshold to avoid division by zero
    if abs(b) > 1e-12:
        ys = -(a*xs + c)/b
    else:
        # vertical horizon (rare) -> use intersections with top/bottom to get x
        # a*x + b*y + c = 0 -> x = -c/a
        x_const = -c/a
        xs = np.array([x_const, x_const])
        ys = np.array([0, H-1], dtype=float)

    plt.figure()
    plt.imshow(im)
    plt.plot(xs, ys, 'r-', linewidth=2)
    plt.title('Horizon line')
    plt.show()

In [19]:
def get_camera_parameters(vpts, image_shape=None):
    """
    Solve for (f, u, v) using SymPy and the three orthogonal vanishing points.

    Inputs:
        vpts : (3,3) ndarray; columns are homogeneous vanishing points for 3 orthogonal world axes
        image_shape : optional (H, W) to seed u,v near image center and f near image size

    Returns:
        f, u, v  (floats)
    """
    V = vpts.astype(float).copy()
    # Normalize each VP to z=1 for numerical stability
    for i in range(3):
        if abs(V[2, i]) > 1e-12:
            V[:, i] /= V[2, i]

    # Symbols
    f, u, v = sp.symbols('f u v', real=True)

    # Intrinsics and IAC
    K = sp.Matrix([[f, 0, u],
                   [0, f, v],
                   [0, 0, 1]])
    Kinv = K.inv()
    omega = Kinv.T * Kinv  # Image of the Absolute Conic

    # Build equations v_i^T ω v_j = 0 for the 3 pairs
    Vsym = [sp.Matrix(V[:, i]) for i in range(3)]
    pairs = [(0,1), (0,2), (1,2)]
    eqs = []
    for i, j in pairs:
        eqs.append(sp.Eq((Vsym[i].T * omega * Vsym[j])[0], 0))

    # Initial guess
    if image_shape is not None:
        H, W = image_shape
        u0 = float(W) / 2.0
        v0 = float(H) / 2.0
        f0 = float(max(H, W))
    else:
        u0, v0, f0 = 800.0, 600.0, 1500.0

    # Use nsolve; try a couple of seeds if needed
    unknowns = (f, u, v)
    seeds = [
        (f0, u0, v0),
        (1.5*f0, u0*0.9, v0*1.1),
        (0.8*f0, u0*1.1, v0*0.9),
    ]
    sol = None
    for s in seeds:
        try:
            sol_candidate = sp.nsolve(eqs, unknowns, s, tol=1e-16, maxsteps=200)
            # Ensure positive focal length
            if float(sol_candidate[0]) > 0:
                sol = sol_candidate
                break
        except Exception:
            continue
    if sol is None:
        # As a last resort, allow negative root then flip sign
        sol = sp.nsolve(eqs, unknowns, seeds[0], tol=1e-16, maxsteps=200)

    f_val, u_val, v_val = map(float, sol)
    f_val = abs(f_val)  # enforce positive focal length
    return f_val, u_val, v_val

In [20]:
def get_rotation_matrix(f=None, u=None, v=None, vpts=None):
    """
    Computes the camera rotation matrix R given intrinsics and vanishing points.
    By definition, direction vectors r_i are proportional to K^{-1} v_i.
    We form columns r1,r2,r3 and orthonormalize by normalization + right-handed fix.
    Inputs:
        f,u,v: intrinsics
        vpts: (3,3) array with columns the three vanishing points
    Returns:
        R: (3,3) rotation matrix
    """
    assert vpts is not None and f is not None and u is not None and v is not None
    # Build K^{-1}
    Kinv = np.array([[1.0/f,     0.0,   -u/f],
                     [0.0,     1.0/f,   -v/f],
                     [0.0,       0.0,    1.0]])
    V = vpts.copy().astype(float)
    for i in range(3):
        if abs(V[2,i]) > 1e-12:
            V[:,i] /= V[2,i]

    Rcols = Kinv @ V  # columns are unnormalized directions
    # Normalize columns
    for i in range(3):
        n = np.linalg.norm(Rcols[:, i])
        if n > 1e-12:
            Rcols[:, i] /= n

    # Enforce orthonormality (small drift): r3 = r1 x r2, then re-orthonormalize
    r1 = Rcols[:,0]
    r2 = Rcols[:,1]
    r3 = np.cross(r1, r2)
    r2 = np.cross(r3, r1)
    r1 /= np.linalg.norm(r1); r2 /= np.linalg.norm(r2); r3 /= np.linalg.norm(r3)
    R = np.stack([r1, r2, r3], axis=1)
    return R

In [21]:
def estimate_height(coord_obj, coord_ref, v_vert, horizon_line, ref_height_m,
                    make_plot=False, im=None, obj_name=""):
    """
    Estimate an object's metric height using a known reference person and the
    vertical vanishing point + horizon line (Criminisi et al.).

    Inputs:
        coord_obj : (3,2) homogeneous pts -> [:,0]=top, [:,1]=bottom of target
        coord_ref : (3,2) homogeneous pts -> [:,0]=top, [:,1]=bottom of reference
        v_vert    : (3,)  vertical vanishing point (homogeneous)
        horizon_line : (3,) ground horizon line (a,b,c) with a^2+b^2=1
        ref_height_m : float, known reference height in meters
        make_plot: bool, optionally draw the construction
        im       : image array, required if make_plot=True
        obj_name : str, label for the plot

    Returns:
        height_m : float, estimated height in meters
    """
    def deh(p):
        return p[:2] / p[2]

    # Ensure homogeneous normalization
    v = v_vert.astype(float)
    if abs(v[2]) > 1e-12:
        v = v / v[2]

    # Lines through the bottoms toward the vertical VP
    L_obj = np.cross(coord_obj[:, 1], v)   # line through object base and vertical VP
    L_ref = np.cross(coord_ref[:, 1], v)   # line through ref base and vertical VP

    # Intersections with horizon: I = ( (base × Vz) × horizon )
    I_obj = np.cross(L_obj, horizon_line)
    I_ref = np.cross(L_ref, horizon_line)

    # Dehomogenize all needed points
    xT, xB = deh(coord_obj[:, 0]), deh(coord_obj[:, 1])
    rT, rB = deh(coord_ref[:, 0]), deh(coord_ref[:, 1])
    I_obj = deh(I_obj)
    I_ref = deh(I_ref)

    # Euclidean distances in image
    def dist(a, b): return float(np.linalg.norm(a - b))
    d_xx   = dist(xT, xB)     # |x x'|
    d_xI   = dist(xB, I_obj)  # |x I|
    d_rI   = dist(rB, I_ref)  # |x0 I0|
    d_rr   = dist(rT, rB)     # |x0 x0'|

    # Cross-ratio based height transfer:
    # H = h * ( |x x'| / |x I| ) * ( |x0 I0| / |x0 x0'| )
    eps = 1e-12
    height_m = ref_height_m * (d_xx / max(d_xI, eps)) * (d_rI / max(d_rr, eps))

    if make_plot and im is not None:
        plt.figure()
        plt.imshow(im)
        # draw vertical rays
        def draw_ray(base):
            # extend to image bounds using intersections with frame
            H, W = im.shape[0], im.shape[1]
            # intersect with left/right borders to make a visible segment
            # line: L_obj or L_ref depending on base
        plt.plot([xB[0], I_obj[0]], [xB[1], I_obj[1]], 'g--', lw=1)
        plt.plot([rB[0], I_ref[0]], [rB[1], I_ref[1]], 'c--', lw=1)
        # segments
        plt.plot([xB[0], xT[0]], [xB[1], xT[1]], 'y-', lw=2, label=f'{obj_name} segment')
        plt.plot([rB[0], rT[0]], [rB[1], rT[1]], 'm-', lw=2, label='reference segment')
        # points
        plt.plot([I_obj[0]], [I_obj[1]], 'ro', label='I (obj)')
        plt.plot([I_ref[0]], [I_ref[1]], 'bo', label='I (ref)')
        plt.title(f'Height construction: {obj_name}  ≈  {height_m:.2f} m')
        plt.legend()
        plt.show()

    return height_m

# Main function

In [22]:
# Load image
im = np.asarray(Image.open('CSL.jpg'))

# Part 1
# Get vanishing points for each of the directions
num_vpts = 3
vpts = np.zeros((3, num_vpts))
saved_lines = []   # keep each direction's lines for plotting/report

for i in range(num_vpts):
    print('Getting vanishing point %d' % i)
    # Get at least three lines from user input
    n, lines, centers = get_input_lines(im)
    saved_lines.append(lines)
    # <YOUR IMPLEMENTATION> Solve for vanishing point
    # Compute vanishing point via SVD
    vpts[:, i] = get_vanishing_point(lines)
    # Plot the lines and the vanishing point
    plot_lines_and_vp(im, lines, vpts[:, i])

# <YOUR IMPLEMENTATION> Get the ground horizon line
horizon_line = get_horizon_line(vpts[:, 0], vpts[:, 1])
# <YOUR IMPLEMENTATION> Plot the ground horizon line
plot_horizon_line(im, horizon_line)

# Part 2
# <YOUR IMPLEMENTATION> Solve for the camera parameters (f, u, v)
f, u, v = get_camera_parameters(vpts, im.shape[:2])
# Part 3
# <YOUR IMPLEMENTATION> Solve for the rotation matrix
R = get_rotation_matrix(f=f, u=u, v=v, vpts=vpts)

# Part 4
# Record image coordinates for each object and store in map
objects = ('person', 'CSL building', 'the spike statue', 'the lamp posts')
coords = dict()
for obj in objects:
    coords[obj] = get_top_and_bottom_coordinates(im, obj)

# <YOUR IMPLEMENTATION> Estimate heights
ref_name = 'person'
ref_height_m = 66 * 0.0254        # 5 ft 6 in  -> meters (change to 72*0.0254 for 6 ft)
v_vert = vpts[:, 2]               # vertical vanishing point

# Initialize a dictionary to store calculated heights
heights = {}

# Loop through all target objects (skip the reference)
for obj in objects[1:]:
    print(f'Estimating height of {obj}')
    height = estimate_height(coords[obj],
                             coords[ref_name],
                             v_vert,
                             horizon_line,
                             ref_height_m,
                             make_plot=True, im=im, obj_name=obj)
    heights[obj] = height   # save each computed height
    print(f'  {obj}: {height:.2f} m  (with {ref_height_m/0.0254:.0f}" reference)')


Getting vanishing point 0
Set at least 3 lines to compute vanishing point
Click the two endpoints, use the right key to undo, and use the middle key to stop input


can't invoke "event" command: application has been destroyed
    while executing
"event generate $w <<ThemeChanged>>"
    (procedure "ttk::ThemeChanged" line 6)
    invoked from within
"ttk::ThemeChanged"


Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Getting vanishing point 1
Set at least 3 lines to compute vanishing point
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input


/tmp/ipykernel_18169/2627096133.py:24: RuntimeWarning: divide by zero encountered in divide
  pt1 = pt1 / pt1[2]
/tmp/ipykernel_18169/2627096133.py:24: RuntimeWarning: invalid value encountered in divide
  pt1 = pt1 / pt1[2]
/tmp/ipykernel_18169/2627096133.py:25: RuntimeWarning: divide by zero encountered in divide
  pt2 = pt2 / pt2[2]
/tmp/ipykernel_18169/2627096133.py:25: RuntimeWarning: invalid value encountered in divide
  pt2 = pt2 / pt2[2]


Getting vanishing point 2
Set at least 3 lines to compute vanishing point
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click on the top coordinate of person
Click on the bottom coordinate of person
Click on the top coordinate of CSL building
Click on the bottom coordinate of CSL building
Click on the top coordinate of the spike statue
Click on the bottom coordinate of the spike statue
Click on the top coordinate of the lamp posts
Click on the bottom coordinate of the lamp posts
Estimating height of CSL building
  CSL building: 29.71 m  (with 66" reference)
Estimating height of the spike statue

In [23]:
# Summary section
print("\n================== REPORT SUMMARY ==================")

# Vanishing points (pixel coordinates)
print("Vanishing points (pixels):")
VPs_px = (vpts / vpts[2])[:2].T  # normalize each so z=1, take x,y only
for i, (x, y) in enumerate(VPs_px):
    print(f"  VP{i}: ({x:.2f}, {y:.2f})")

# Horizon line parameters
a, b, c = horizon_line
print("\nGround horizon line (normalized so a^2 + b^2 = 1):")
print(f"  a*x + b*y + c = 0")
print(f"  a = {a:.6f}, b = {b:.6f}, c = {c:.6f}")
print(f"  (a^2 + b^2) = {a*a + b*b:.3f}")

# Camera parameters
print("\nCamera intrinsic parameters:")
print(f"  f = {f:.3f} px")
print(f"  u = {u:.3f} px")
print(f"  v = {v:.3f} px")

# Rotation matrix
print("\nRotation matrix R:")
np.set_printoptions(precision=6, suppress=True)
print(R)

# Object heights using 5'6" reference
print("\nEstimated heights (reference = 5 ft 6 in):")
for obj, h in heights.items():
    print(f"  {obj}: {h:.2f} m ({h*3.28084:.1f} ft)")

# Recalculate if reference = 6 ft
scale = 72 / 66
print("\nIf reference person is 6 ft tall:")
for obj, h in heights.items():
    h_scaled = h * scale
    print(f"  {obj}: {h_scaled:.2f} m ({h_scaled*3.28084:.1f} ft)")

print("====================================================\n")



================== REPORT SUMMARY ==================
Vanishing points (pixels):
  VP0: (-345.65, 219.09)
  VP1: (1363.52, 234.75)
  VP2: (953.53, -31628.54)

Ground horizon line (normalized so a^2 + b^2 = 1):
  a*x + b*y + c = 0
  a = -0.009165, b = 0.999958, c = -222.248090
  (a^2 + b^2) = 1.000

Camera intrinsic parameters:
  f = 840.593 px
  u = 661.754 px
  v = 206.127 px

Rotation matrix R:
[[-0.767775  0.640654 -0.009162]
 [ 0.009879  0.026135  0.99961 ]
 [ 0.640643  0.767385 -0.026395]]

Estimated heights (reference = 5 ft 6 in):
  CSL building: 29.71 m (97.5 ft)
  the spike statue: 13.21 m (43.4 ft)
  the lamp posts: 5.23 m (17.1 ft)

If reference person is 6 ft tall:
  CSL building: 32.41 m (106.3 ft)
  the spike statue: 14.42 m (47.3 ft)
  the lamp posts: 5.70 m (18.7 ft)



## Extra-Credit: Additional Measurements

For extra credit, I performed additional measurements on the image to answer questions such as:  
- Which of the people visible are the tallest?  
- What are the heights of the windows?  

Below are the functions and code used for these measurements.

In [32]:
def _midpoint(p_top, p_bot):
    """
    Calculate the midpoint between two points.

    Args:
        p_top: Tuple or array-like, coordinates of the top point (x, y).
        p_bot: Tuple or array-like, coordinates of the bottom point (x, y).

    Returns:
        Tuple (x, y) representing the midpoint.
    """
    # Compute average of x and y coordinates
    return ((p_top[0]+p_bot[0])*0.5, (p_top[1]+p_bot[1])*0.5)

invalid command name "135331422620224delayed_destroy"
    while executing
"135331422620224delayed_destroy"
    ("after" script)
invalid command name "135331320873664delayed_destroy"
    while executing
"135331320873664delayed_destroy"
    ("after" script)
invalid command name "135331320857856delayed_destroy"
    while executing
"135331320857856delayed_destroy"
    ("after" script)
invalid command name "135331399346048delayed_destroy"
    while executing
"135331399346048delayed_destroy"
    ("after" script)
invalid command name "135331368907520delayed_destroy"
    while executing
"135331368907520delayed_destroy"
    ("after" script)
invalid command name "135331423408832delayed_destroy"
    while executing
"135331423408832delayed_destroy"
    ("after" script)
invalid command name "135331421267584delayed_destroy"
    while executing
"135331421267584delayed_destroy"
    ("after" script)
invalid command name "135331425058688delayed_destroy"
    while executing
"135331425058688delayed_destro

In [ ]:
def plot_labeled_segments(im, coords_dict, title="Labeled segments",
                          colors=None, highlight_label=None,
                          save_path=None):
    """
    Draw top-to-bottom segments for each object/person in the image, with text labels at midpoints.

    Args:
        im: RGB image array (numpy array).
        coords_dict: dict[label] -> (3,2) homogeneous coordinates, [:,0]=top, [:,1]=bottom.
        title: Title for the plot (string).
        colors: Optional dict[label] -> matplotlib color string.
        highlight_label: Optional label to highlight (thicker line, different color).
        save_path: Optional file path to save the figure (string).

    Returns:
        None. Displays and optionally saves the plot.
    """
    plt.figure(figsize=(8,6))
    plt.imshow(im)
    ax = plt.gca()

    for lab, coord in coords_dict.items():
        # Dehomogenize top and bottom points
        top = (coord[0,0]/coord[2,0], coord[1,0]/coord[2,0])
        bot = (coord[0,1]/coord[2,1], coord[1,1]/coord[2,1])
        mid = _midpoint(top, bot)

        # Highlight if this is the tallest (or specified) label
        is_hi = (lab == highlight_label)
        clr   = (colors.get(lab) if colors and lab in colors
                 else ('tab:red' if is_hi else 'tab:blue'))
        lw    = 3 if is_hi else 2
        alpha = 0.95 if is_hi else 0.85

        # Draw segment and endpoints
        ax.plot([top[0], bot[0]], [top[1], bot[1]], '-', color=clr, lw=lw, alpha=alpha)
        ax.plot([top[0], bot[0]], [top[1], bot[1]], 'o', color=clr, ms=3, alpha=alpha)
        # Add label near midpoint
        ax.text(mid[0], mid[1]-8, lab, color=clr, fontsize=11,
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.6, boxstyle='round,pad=0.2'))

    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def estimate_window_height_by_local_ratio(coord_win, coord_floor, h_floor_m):
    """
    Estimate the height of a window by comparing its pixel height to a reference floor segment.

    Args:
        coord_win: (3,2) homogeneous coordinates for the window ([:,0]=top, [:,1]=bottom).
        coord_floor: (3,2) homogeneous coordinates for the reference floor segment.
        h_floor_m: Metric height (meters) of the reference floor segment.

    Returns:
        Estimated window height in meters (float).
    """
    def deh(c): return (c[:2] / c[2]).T  # Dehomogenize to get 2D coordinates
    w = deh(coord_win)
    f = deh(coord_floor)
    # Calculate pixel heights
    px_win   = float(np.linalg.norm(w[:,0] - w[:,1]))
    px_floor = float(np.linalg.norm(f[:,0] - f[:,1])) + 1e-12  # Avoid division by zero
    # Scale window height by reference floor height
    return h_floor_m * (px_win / px_floor)

In [ ]:
def plot_people_with_tallest(im, coords_people, coord_ref, v_vert, horizon_line,
                             ref_height_m, title="People (labeled, tallest highlighted)",
                             save_path=None):
    """
    Estimate heights for each person, print a sorted list, and plot them with the tallest highlighted.

    Args:
        im: RGB image array (numpy array).
        coords_people: dict[label] -> (3,2) homogeneous coordinates for people.
        coord_ref: (3,2) homogeneous coordinates for the reference person.
        v_vert: (3,) vertical vanishing point (homogeneous).
        horizon_line: (3,) ground horizon line parameters.
        ref_height_m: Reference person height in meters.
        title: Title for the plot (string).
        save_path: Optional file path to save the figure (string).

    Returns:
        Tuple:
            heights: dict[label] -> estimated heights in meters.
            tallest_label: label of the tallest person.
    """
    # 1) Compute heights for each person
    heights = {}
    for lab, coord in coords_people.items():
        h = estimate_height(coord, coord_ref, v_vert, horizon_line, ref_height_m,
                            make_plot=False, im=None, obj_name=lab)
        heights[lab] = h

    # 2) Find the tallest person
    tallest_label = max(heights.items(), key=lambda kv: kv[1])[0]

    # 3) Print summary of heights
    print("People heights (descending):")
    for lab, h in sorted(heights.items(), key=lambda kv: kv[1], reverse=True):
        print(f"  {lab}: {h:.2f} m ({h*3.28084:.1f} ft)")
    print(f"\nTallest: {tallest_label}  → {heights[tallest_label]:.2f} m")

    # 4) Plot with tallest highlighted
    plot_labeled_segments(im, coords_people, title=title,
                          highlight_label=tallest_label, save_path=save_path)

    return heights, tallest_label

In [ ]:
def click_segments(im, labels, title="Click TOP then BOTTOM for each label (Right-click=undo, Middle-click=stop)"):
    """
    Collect top/bottom homogeneous points for a list of targets in one shot.

    Args:
        im: RGB image array (numpy array).
        labels: List or tuple of labels (strings) for each target.
        title: Instructions for the user (string).

    Returns:
        Dictionary mapping label -> (3,2) array: [:,0]=top, [:,1]=bottom.
    """
    out = {}
    # Iterate over each label in the provided list
    for label in labels:
        # Prompt user to click top and bottom coordinates for the current label
        coord = get_top_and_bottom_coordinates(im, label)
        # Store the coordinates in the output dictionary
        out[label] = coord
    # Return the dictionary mapping labels to their coordinates
    return out

### People Height Measurement Workflow

- **click_segments:**  
  Collects top and bottom coordinates for each person via user clicks.

- **plot_people_with_tallest:**  
  Estimates heights for all labeled people using projective geometry, prints a sorted summary, and highlights the tallest in the plot.

- **estimate_height:**  
  Used internally to compute each person's metric height from image coordinates and geometric parameters.

These functions work together to measure, compare, and visualize the heights of people in the image.

In [34]:
# Define labels for the people to be measured
people_labels = ("Person A", "Person B", "Person C")

# Collect top and bottom coordinates for each person via user clicks
coords_people = click_segments(im, people_labels)

# Estimate heights for each person and plot them, highlighting the tallest
people_heights, tallest_name = plot_people_with_tallest(
    im,                        # input image
    coords_people,             # dictionary of coordinates for each person
    coord_ref=coords['person'],# reference person coordinates for metric scaling
    v_vert=v_vert,             # vertical vanishing point
    horizon_line=horizon_line, # ground horizon line
    ref_height_m=ref_height_m, # reference person height in meters
    title="People (labeled, tallest highlighted)", # plot title
    save_path="people_tallest.png",                # file to save the plot
)

Click on the top coordinate of Person A
Click on the bottom coordinate of Person A
Click on the top coordinate of Person B
Click on the bottom coordinate of Person B
Click on the top coordinate of Person C
Click on the bottom coordinate of Person C
People heights (descending):
  Person A: 1.76 m (5.8 ft)
  Person C: 1.75 m (5.8 ft)
  Person B: 1.69 m (5.6 ft)

Tallest: Person A  → 1.76 m


### Window Height Measurement Workflow

- **click_segments:**  
  Collects top and bottom coordinates for each window via user clicks.

- **plot_labeled_windows_with_stats:**  
  Estimates heights for all labeled windows, prints a summary, and visualizes the results.

- **estimate_window_height_by_local_ratio / estimate_height:**  
  Used internally to compute each window's metric height, either by local ratio (using a reference floor segment) or ground-based geometry.

These functions work together to measure and visualize the heights of windows in the image.

In [ ]:
def plot_labeled_windows_with_stats(
    im, coords_windows, coord_ref, v_vert, horizon_line, ref_height_m,
    title="Windows (labeled)", save_path=None,
    coord_floor=None, h_floor_m=None
):
    """
    Estimate and plot the heights of windows, using either a local ratio (if a reference floor segment is provided)
    or the ground-based method.

    Args:
        im: RGB image array (numpy array).
        coords_windows: dict[label] -> (3,2) homogeneous coordinates for windows.
        coord_ref: (3,2) homogeneous coordinates for the reference object (e.g., person).
        v_vert: (3,) vertical vanishing point (homogeneous).
        horizon_line: (3,) ground horizon line parameters.
        ref_height_m: Reference object height in meters.
        title: Title for the plot (string).
        save_path: Optional file path to save the figure (string).
        coord_floor: Optional (3,2) homogeneous coordinates for a grounded floor segment.
        h_floor_m: Optional metric height (meters) of the grounded floor segment.

    Returns:
        Dictionary mapping window labels to their estimated heights in meters.
    """
    # If user provided a grounded floor segment but not its metric height, compute it once.
    if coord_floor is not None and h_floor_m is None:
        h_floor_m = estimate_height(coord_floor, coord_ref, v_vert, horizon_line, ref_height_m,
                                    make_plot=False, im=None, obj_name="floor")

    heights = {}
    for lab, coord in coords_windows.items():
        # Use local ratio if possible, otherwise fallback to ground-based method
        if coord_floor is not None and h_floor_m is not None:
            h = estimate_window_height_by_local_ratio(coord, coord_floor, h_floor_m)
        else:
            # Fallback (only valid if segment base is on the ground plane)
            h = estimate_height(coord, coord_ref, v_vert, horizon_line, ref_height_m,
                                make_plot=False, im=None, obj_name=lab)
        heights[lab] = h

    # Print stats
    vals = list(heights.values())
    if vals:
        avg_h = sum(vals)/len(vals)
        print("Window heights:")
        for lab in sorted(heights, key=heights.get, reverse=True):
            h = heights[lab]
            print(f"  {lab}: {h:.2f} m ({h*3.28084:.1f} ft)")
        print(f"\nAverage window height: {avg_h:.2f} m ({avg_h*3.28084:.1f} ft)")

    # Plot labeled segments for windows
    plot_labeled_segments(im, coords_windows, title=title, save_path=save_path)
    return heights

In [33]:
# One-time click: get a grounded vertical segment spanning one floor on the façade
coord_floor = click_segments(im, ["First Floor (Grounded)"])["First Floor (Grounded)"]

# 1) Collect top and bottom coordinates for each window via user clicks
window_labels = ("Window 1", "Window 2", "Window 3")
coords_windows = click_segments(im, window_labels)

# Estimate window heights using the reference floor segment
# The function computes the metric height of the floor segment internally
window_heights = plot_labeled_windows_with_stats(
    im,                             # input image
    coords_windows,                 # dictionary of coordinates for each window
    coord_ref=coords['person'],     # reference person coordinates for metric scaling
    v_vert=v_vert,                  # vertical vanishing point
    horizon_line=horizon_line,      # ground horizon line
    ref_height_m=ref_height_m,      # reference person height in meters
    title="Windows (labeled)",      # plot title
    save_path="window_heights.png", # file to save the plot
    coord_floor=coord_floor         # reference floor segment for local ratio method
)

Click on the top coordinate of First Floor (Grounded)
Click on the bottom coordinate of First Floor (Grounded)
Click on the top coordinate of Window 1
Click on the bottom coordinate of Window 1
Click on the top coordinate of Window 2
Click on the bottom coordinate of Window 2
Click on the top coordinate of Window 3
Click on the bottom coordinate of Window 3
Window heights:
  Window 3: 4.16 m (13.6 ft)
  Window 2: 3.85 m (12.6 ft)
  Window 1: 3.48 m (11.4 ft)

Average window height: 3.83 m (12.6 ft)
